# Train on 221 EMNLP Videos - XGBoost Baseline
Features: WavLM 768-dim + prosody 23-dim = 791 total
Labels: EMNLP word-level BIO tags (B/I/L = laugh)

In [ ]:
# Setup
from google.colab import drive
drive.mount('/content/drive')

import os, numpy as np, pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.linear_model import LogisticRegression
import xgboost as xgb

BASE = '/content/drive/MyDrive/standup4ai'
FEAT_DIR = BASE + '/features_221'
LABEL_DIR = BASE + '/seq-Standup4AI/dataset/en_uk/emnlp+jahak/train'

print('Setup complete')


In [ ]:
# Load with timestamp-based chunk mapping
feat_files = sorted([f for f in os.listdir(FEAT_DIR) if f.endswith('_features.npy')])
print('Feature files:', len(feat_files))

def parse_timestamp(ts_str):
    ts_str = str(ts_str).strip()
    try:
        parts = ts_str.strip('[]').split(',')
        return float(parts[0]), float(parts[1])
    except:
        return None, None

X_list, y_list, vids = [], [], []
missing = 0

for fi, f in enumerate(feat_files):
    vid = f.replace('_features.npy', '')
    label_path = LABEL_DIR + '/' + vid + '.csv'
    if not os.path.exists(label_path):
        missing += 1
        continue
    
    feats = np.load(FEAT_DIR + '/' + f)
    labels_df = pd.read_csv(label_path)
    n_chunks = len(feats)
    chunk_dur = 5.0
    
    word_times, word_labels = [], []
    for _, row in labels_df.iterrows():
        t0, t1 = parse_timestamp(row['timestamp'])
        if t0 is not None:
            word_times.append((t0, t1))
            word_labels.append(str(row['label']).strip())
    
    chunk_labels = []
    for i in range(n_chunks):
        c0, c1 = i * chunk_dur, (i+1) * chunk_dur
        is_laugh = False
        for (w0, w1), wl in zip(word_times, word_labels):
            if wl in ['B', 'I', 'L'] and w0 < c1 and w1 > c0:
                is_laugh = True
                break
        chunk_labels.append(1 if is_laugh else 0)
    
    X_list.append(feats)
    y_list.append(np.array(chunk_labels, dtype=np.float32))
    vids.extend([vid] * n_chunks)

X = np.vstack(X_list)
y = np.concatenate(y_list)
groups = np.array(vids)

print('X:', X.shape, 'y:', y.shape, 'pos rate:', round(y.mean(), 3))
print('Unique videos:', len(set(vids)), 'missing:', missing)

# Stats
print('y distribution:', np.bincount(y.astype(int)))


In [ ]:
# Try multiple models

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

gkf = GroupKFold(n_splits=5)

# Model 1: Logistic Regression (baseline)
print('=== Logistic Regression ===')
lr_f1s = []
for fold, (tr_idx, te_idx) in enumerate(gkf.split(X_scaled, y, groups)):
    Xtr, Xte = X_scaled[tr_idx], X_scaled[te_idx]
    ytr, yte = y[tr_idx], y[te_idx]
    
    m = LogisticRegression(C=0.1, max_iter=500, class_weight='balanced')
    m.fit(Xtr, ytr)
    probs = m.predict_proba(Xte)[:, 1]
    f = f1_score(yte, (probs >= 0.5).astype(int), zero_division=0)
    lr_f1s.append(f)
    print(f'  Fold {fold+1}: F1={f:.4f}')
print(f'  LR CV F1: {np.mean(lr_f1s):.4f} +/- {np.std(lr_f1s):.4f}')

# Model 2: XGBoost
print('')
print('=== XGBoost ===')
xgb_f1s = []
for fold, (tr_idx, te_idx) in enumerate(gkf.split(X_scaled, y, groups)):
    Xtr, Xte = X_scaled[tr_idx], X_scaled[te_idx]
    ytr, yte = y[tr_idx], y[te_idx]
    
    # Compute scale_pos_weight for imbalance
    neg = (ytr == 0).sum()
    pos = (ytr == 1).sum()
    scale = neg / max(pos, 1)
    
    m = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.05,
        scale_pos_weight=scale,
        use_label_encoder=False,
        eval_metric='logloss',
        verbosity=0
    )
    m.fit(Xtr, ytr)
    probs = m.predict_proba(Xte)[:, 1]
    f = f1_score(yte, (probs >= 0.5).astype(int), zero_division=0)
    xgb_f1s.append(f)
    p = precision_score(yte, (probs >= 0.5).astype(int), zero_division=0)
    r = recall_score(yte, (probs >= 0.5).astype(int), zero_division=0)
    print(f'  Fold {fold+1}: F1={f:.4f} P={p:.4f} R={r:.4f} (scale={scale:.2f})')
print(f'  XGB CV F1: {np.mean(xgb_f1s):.4f} +/- {np.std(xgb_f1s):.4f}')

# Save results
results = {
    'lr_cv_f1': float(np.mean(lr_f1s)),
    'xgb_cv_f1': float(np.mean(xgb_f1s)),
    'lr_std': float(np.std(lr_f1s)),
    'xgb_std': float(np.std(xgb_f1s)),
    'pos_rate': float(y.mean()),
    'n_videos': int(len(set(vids)))
}
import json
with open(BASE + '/training_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('')
print('Results:', results)
